# Algoritmo di correzione automatica per un motore di ricerca

In questo progetto viene implementato un sistema di correzione automatica delle query di ricarca basato sulla distanza di Levenshtein.

* Costruzione del dizionario di base
* Implementazione dell'algoritmo di Levenshtein
* Funzione di correzione principale
* Funzione per la visualizzazione dei risultati
* Implementazione test
* Implementazione modalità interattiva
* Esecuzione test
* Esecuzione modalità interattiva

In [1]:
#DIZIONARIO DI BASE

#il dizionario simula il vocabolario di un motore di ricerca

DIZIONARIO_BASE= [

    "rapporto", "relazione", "documento", "documentazione", "archivio",
    "fascicolo", "verbale", "contratto", "fattura", "preventivo",

    "vendite", "acquisti", "marketing", "finanza", "contabilita",
    "amministrazione", "risorse", "umane", "produzione", "logistica",
    "magazzino", "spedizione", "assistenza", "clienti", "fornitori",

    "annuale", "mensile", "trimestrale", "settimanale", "giornaliero",
    "bilancio", "budget", "previsione", "consuntivo", "analisi",

    "gennaio", "febbraio", "marzo", "aprile", "maggio", "giugno",
    "luglio", "agosto", "settembre", "ottobre", "novembre", "dicembre",

    "progetto", "obiettivo", "strategia", "risultato", "performance",
    "riunione", "presentazione", "proposta", "approvazione", "revisione",
    "aggiornamento", "procedura", "manuale", "istruzioni", "normativa",

    "fatturato", "profitto", "perdita", "investimento", "costo",
    "ricavo", "margine", "sconto", "pagamento", "rimborso",
]

In [2]:
#IMPLEMENTAZIONE DELL'ALGORITMO DI LEVENSHTEIN

def levenshtein_distance(parola1,parola2):
    """
    calcola la distanza di levenshtein tra due stringhe.

    La distanza di levenshtein misura il numero minimo di operazioni elementari 
    (inserimento, cancellazione, sostituzione di un carattere) necessarie per trasformare 
    una stringa nell'altra

    i parametri sono le due stringhe da confrontare: parola1 e parola2

    la funzione ritorna un valore intero che è il numero minimo di operazioni per trasformare parola1 in parola2
    """

    #lunghezza delle due stringhe
    n=len(parola1)
    m=len(parola2)

    #casi banali (stringhe vuote)

    #se la prima parola è vuota allora l'unica soluzione è inserire tutti i caratteri della
    #seconda parola, quindi distanza = lunghezza di parola2. Viceversa se la seconda parola è vuota

    if n==0:
        return m 
    if m==0:
        return n

    #costruzione di una matrice dinamica

    #matrice (n+1) x (m+1) dove ogni cella matrix[i][j] rappresenterà la distanza di levenshtein 
    #tra i primi i caratteri di parola1 e i primi j caratteri di parola2

    matrix=[[0]*(m+1) for _ in range (n+1)] # inizializzazione matrice con 0 ovunque

    #inizializzazione della prima colonna: trasformare i caratteri di parola1 in stringa vuota
    #richiede cancellazioni (una per ogni carattere)
    for i in range(n + 1): 
        matrix[i][0] = i  # costo di cancellare i caratteri

    #inizializzazione della prima riga: trasformare una stringa vuota in j caratteri di parola2
    #richiede j inserimenti
    for j in range(m + 1):
        matrix[0][j] = j  # Costo di inseririre j caratteri
    
    #riempimento della matrice

    #scorrimento di tutte le combinazioni di caratteri tra le due parole
    for i in range(1, n + 1): #per ogni carattere di parola1
        for j in range(1, m + 1): #per ogni carattere di parola2

            #confronto dei caratteri correnti (se i caratteri sono uguali allora non servono operazioni)
            if parola1[i - 1] == parola2[j - 1]:
                costo_sostituzione = 0  #caratteri identici allora costo zero
            else:
                costo_sostituzione = 1  #caratteri diversi allora costo di sostituzione

            #calcolo del minimo tra tre possibili operazioni:
            #matrix[i-1][j] + 1 allora cancellazione del carattere in parola1
            #matrix[i][j-1] + 1 allora inserimento di un carattere in parola1
            #matrix[i-1][j-1] + costo allora sostituzione (o niente se uguali)
            matrix[i][j] = min(
                matrix[i - 1][j] + 1, #cancellazione
                matrix[i][j - 1] + 1, #inserimento
                matrix[i - 1][j - 1] + costo_sostituzione #sostituzione
            )

    #distanza finale tra le due parole complete (nell'angolo in basso a destra)
    return matrix[n][m]
    

In [3]:
#FUNZIONE DI CORREZIONE PRINCIPALE

def suggest_correction(query, dictionary):
    """
    questa è la funzione principale per la correzione automatica di una query di ricerca

    per ogni parola della query, cerca nel dizionario la parola più simile
    usando la distanza di Levenshtein. Se la parola è già nel dizionario
    o la distanza minima supera una soglia allora la parola viene lasciata invariata.

    parametri: query (stringa di ricerca inserita dall'utente) e dictionary (ista di parole valide 
    nel vocabolario di riferimento)

    la funzione ritorna una stringa che rappresenta la query corretta (con le parole corrette al posto 
    di quelle errate) oppure restituisce la query originale se è già valida o non correggibile
    """

    #conversione della query in minuscolo per un confronto case-insensitive e divisione in parole 
    #singole usando gli spazi come separatori

    parole_query = query.lower().split()

    #lista che conterrà le parole corrette (o originali) per ogni token

    parole_corrette = []

    #soglia di correzione
    #definizione di una soglia massima di distanza oltre la quale non viene suggerita alcuna correzione: se 
    #nessuna parola del dizionario è abbastanza vicina allora viene mantenuta la parola originale 
    #la soglia è calcolata dinamicamente in base alla lunghezza della parola:
    #parole corte (<=3 lettere) allora soglia 1
    #parole medie (4-6 lettere) allora soglia 2
    #parole lunghe (>6 lettere) allora soglia 3

    #iterazione su ogni parola presente nella query
    for parola in parole_query:

        #controllo: la parola è già nel dizionario? se la parola digitata è già valida allora non serve 
        #alcuna correzione
        if parola in dictionary:
            parole_corrette.append(parola) #aggiunge la parola così come è
            continue #passa alla parola successiva

        #calcolo della soglia dinamica
        lunghezza = len(parola)  #numero di caratteri della parola corrente

        if lunghezza <= 3:
            soglia = 1 #parole brevi (tollera massimo 1 operazione di modifica)
        elif lunghezza <= 6:
            soglia = 2 #parole medie (tollera massimo 2 operazioni)
        else:
            soglia = 3 #parole lunghe (tollera massimo 3 operazioni)

        #ricerca della parola più simile nel dizionario
        parola_migliore = None   #mantiene la parola del dizionario più vicina
        distanza_minima = float('inf')  #inizializza con "infinito" per trovare il minimo

        #scorre ogni parola del dizionario e calcola la distanza di levenshtein
        for termine in dictionary:
            distanza = levenshtein_distance(parola, termine)  #calcolo distanza

            #se la distanza è la più piccola trovata finora allora aggiorna il candidato
            if distanza < distanza_minima:
                distanza_minima = distanza #aggiorna la distanza minima
                parola_migliore = termine #aggiorna la parola candidata

        #decisione
        if distanza_minima <= soglia and parola_migliore is not None:
            #se la parola del dizionario è sufficientemente vicina allora viene usata come correzione
            parole_corrette.append(parola_migliore)
        else:
            #se nessuna parola è abbastanza vicina allora viene mantenuta la parola originale
            parole_corrette.append(parola)

    #ricostruzione della query completa unendo le parole corrette con uno spazio
    query_corretta = " ".join(parole_corrette)

    return query_corretta

In [4]:
#FUNZIONE PER LA VISUALIZZAZIONE DEI RISULTATI
def mostra_risultato(query_originale, query_corretta):
    """
    mostra a schermo il risultato della correzione in modo leggibile

    parametri: query_originale (la query inserita dall'utente) e query_corretta (la query dopo
    la correzione automatica)
    """

    print(f"Input: '{query_originale}'")

    #controlla se la correzione ha modificato qualcosa
    if query_originale.lower() == query_corretta:
        print(f"Output: '{query_corretta}' >> nessuna correzione necessaria") #la query era già corretta quindi nessuna modifica
    else:
        print(f" Output: '{query_corretta}' >> correzione applicata") #la query è stata corretta: quindi mostra la versione modificata



In [ ]:
#IMPLEMENTAZIONE DEI TEST

def esegui_test():
    """
    esegue alcuni casi di test per verificare il corretto funzionamento dell'algoritmo

    i casi di test coprono: errori di battitura comuni, query multi-parola con più errori, query già 
    corrette, parole non correggibili (come ad esempio nomi propri, anni, codici)
    """

    print("Test dell'algoritmo di correzione automatica")
    print()

    #definizione dei casi di test come lista di tuple (query_input, descrizione_caso)
    casi_test = [

        ("raporto vendite",
         "Test 1: lettera mancante ('raporto' >> 'rapporto')"),

        ("analizi trimestrale",
         "Test 2: lettera scambiata ('analizi' >> 'analisi')"),

        ("progettto marketing",
         "Test 3: lettera duplicata ('progettto' >> 'progetto')"),

        ("bilancoi annuale",
         "Test 4: lettere invertite ('bilancoi' >> 'bilancio')"),

        ("preventivoo fornitori",
         "Test 5: lettera finale doppia ('preventivoo' >> 'preventivo')"),

        #query multi-parola con errori multipli
        ("raporto vendite mensille",
         "Test 6: due errori in query multi-parola"),

        ("rapporto vendite annuale",
         "Test 7: query già corretta (nessuna modifica attesa)"),

        #parola corta con un errore 
        ("costo margune",
         "Test 8: errore in parola media ('margune' >> 'margine')"),

        #query con anno (numero non correggibile)
        ("fattura 2023",
         "Test 9: parola mista e anno non correggibile, 'fattura' corretta"),

        ("administratione risorse",
         "Test 10: errore iniziale ('administratione' >> 'amministrazione')"),

        #parola molto distorta (fuori soglia e quindi nessuna correzione)
        ("xqzwrt vendite",
         "Test 11: parola incomprensibile (fuori soglia, quindi no correzione)"),

        #query con solo parole corrette più lunga
        ("relazione trimestrale marketing",
         "Test 12: query lunga già corretta"),
    ]

    #itera su ogni caso di test ed esegue la correzione
    for i, (query, descrizione) in enumerate(casi_test, start=1):

        print(f"\n{descrizione}")#stampa la descrizione del test

        #chiama la funzione principale di correzione
        risultato = suggest_correction(query, DIZIONARIO_BASE)

        #mostra l'input originale e l'output corretto
        mostra_risultato(query, risultato)

    print("\nFine dei test. Algoritmo verificato su", len(casi_test), "casi.")

In [ ]:
#IMPLEMENTAZIONE DI UNA MODALITA' INTERATTIVA

def modalita_interattiva():
    """
    avvia una sessione interattiva in cui l'utente può inserire query e ricevere in tempo reale 
    i suggerimenti di correzione

    la sessione termina digitando 'exit' o 'quit'
    """

    print("\nModalità interattiva")
    print("digitare una query per ricevere la correzione automatica. Digita 'exit' per uscire")

    while True:
        query = input("Cerca: ").strip() #legge l'input dell'utente dal terminale

        if query.lower() in ("exit", "quit", ""): #condizione di uscita: l'utente digita 'exit' o 'quit'
            print("\nModalità interattiva terminata")
            break

        #esecuzione della correzione sulla query inserita
        risultato = suggest_correction(query, DIZIONARIO_BASE)

        #mostra risultato
        mostra_risultato(query, risultato)

In [8]:
#ESECUZIONE TEST
if __name__ == "__main__":
    esegui_test()

Test dell'algoritmo di correzione automatica


Test 1: lettera mancante ('raporto' >> 'rapporto')
Input: 'raporto vendite'
 Output: 'rapporto vendite' >> correzione applicata

Test 2: lettera scambiata ('analizi' >> 'analisi')
Input: 'analizi trimestrale'
 Output: 'analisi trimestrale' >> correzione applicata

Test 3: lettera duplicata ('progettto' >> 'progetto')
Input: 'progettto marketing'
 Output: 'progetto marketing' >> correzione applicata

Test 4: lettere invertite ('bilancoi' >> 'bilancio')
Input: 'bilancoi annuale'
 Output: 'bilancio annuale' >> correzione applicata

Test 5: lettera finale doppia ('preventivoo' >> 'preventivo')
Input: 'preventivoo fornitori'
 Output: 'preventivo fornitori' >> correzione applicata

Test 6: due errori in query multi-parola
Input: 'raporto vendite mensille'
 Output: 'rapporto vendite mensile' >> correzione applicata

Test 7: query già corretta (nessuna modifica attesa)
Input: 'rapporto vendite annuale'
Output: 'rapporto vendite annuale' >> nessuna

In [10]:
#ESECUZIONE MODALITA' INTERATTIVA
if __name__ == "__main__":
    modalita_interattiva()


Modalità interattiva
digirate una query per ricevere la correzione automatica. Digita 'exit' per uscire
Input: 'Blancio'
 Output: 'bilancio' >> correzione applicata

Modalità interattiva terminata
